# Baseline — AML 2026 Task 1: brain age from MRI features

This notebook is the quickest way into the project. The full, reproducible pipeline
lives in the `brainage/` package and is driven by `solution.py` (see `README.md`).

Pipeline: `VarianceThreshold → MADOutlierMasker(5σ) → median impute → QuantileTransformer(normal)
→ SelectKBest(f_regression, k=180) → stack(SVR + GP + XGB, RidgeCV meta)`. Every step is refit inside each CV fold.
Metric: R², RepeatedKFold(5×3, seed 42). Final (tuned) model: **CV R² = 0.5812 ± 0.0331**.

Run everything end-to-end (CV + submission) from the terminal:
```
python solution.py
```


In [3]:
from brainage.config import Config
from brainage.data import load_data, write_submission
from brainage.models import build_model
from brainage.evaluate import cross_validate
import numpy as np

cfg = Config()                        # all paths / seed / hyperparameters in one place
data = load_data(cfg)                 # validated load of data/X_train.csv, y_train.csv, X_test.csv
model = build_model(cfg.model, cfg)   # cfg.model = 'stack2' (use 'vote', 'svr', 'xgb', ... to compare)

scores = cross_validate(model, data.X, data.y, cfg)
print(f'CV R^2 = {scores.mean():.4f} +/- {scores.std():.4f}  ({len(scores)} folds)')

model.fit(data.X, data.y)
pred = np.clip(model.predict(data.X_test), data.y.min(), data.y.max())
write_submission(cfg.submission_path, data.test_ids, pred, cfg.sample_path)
print(f'wrote {cfg.submission_path}')


CV R^2 = 0.5728 +/- 0.0348  (15 folds)


FileNotFoundError: [Errno 2] No such file or directory: '/Users/nathanchallande/Documents/ETHZ/AML/AGE_BRAIN_MRI/sample.csv'